Predicting Feature similarity and adaption weights

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import optuna
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

df = sns.load_dataset('mpg').dropna()
feature_cols = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
num_features = len(feature_cols)

X_raw = df[feature_cols].values
y_raw = df['mpg'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

folds = []
for train_idx, test_idx in kf.split(X_scaled):
    X_tr, y_tr = X_scaled[train_idx], y_raw[train_idx]
    X_te, y_te = X_scaled[test_idx], y_raw[test_idx]
    
    diff_tr = X_tr[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    diff_te = X_te[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    
    folds.append({
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_te': X_te, 'y_te': y_te,
        'diff_tr_sq': diff_tr ** 2,
        'diff_te_sq': diff_te ** 2
    })

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_sim(trial):
    weights = np.array([trial.suggest_float(f'w_{col}', 0.0, 1.0) for col in feature_cols])
    
    w_sum = np.sum(weights)
    if w_sum == 0: return float('inf')
    weights = weights / w_sum

    mean_train_mse = 0
    for fold in folds:
        dist = np.sum(weights * fold['diff_tr_sq'], axis=2)
        np.fill_diagonal(dist, np.inf)
        
        nn_indices = np.argmin(dist, axis=1)
        preds = fold['y_tr'][nn_indices]
        mean_train_mse += mean_squared_error(fold['y_tr'], preds)
        
    return mean_train_mse / 10

print("Starting Phase 1: Optuna optimization for Similarity Weights (10-fold Avg LOOCV)")
study_sim = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_sim.optimize(objective_sim, n_trials=10000)

best_w_raw = np.array([study_sim.best_params[f'w_{col}'] for col in feature_cols])
best_w = best_w_raw / np.sum(best_w_raw)
print(f"Phase 1 Complete. Best 10-Fold Avg LOOCV MSE: {study_sim.best_value:.4f}\n")

for fold in folds:
    dist_train = np.sum(best_w * fold['diff_tr_sq'], axis=2)
    np.fill_diagonal(dist_train, np.inf)
    
    nn_idx = np.argmin(dist_train, axis=1)
    fold['nn_idx_tr'] = nn_idx
    fold['retrieved_X_tr'] = fold['X_tr'][nn_idx]
    fold['retrieved_y_tr'] = fold['y_tr'][nn_idx]

def objective_adapt(trial):
    a = np.array([trial.suggest_float(f'a_{col}', -10.0, 10.0) for col in feature_cols])
    
    mean_train_mse = 0
    for fold in folds:
        adaptation = np.sum(a * (fold['retrieved_X_tr'] - fold['X_tr']), axis=1)
        preds = fold['retrieved_y_tr'] + adaptation
        mean_train_mse += mean_squared_error(fold['y_tr'], preds)
        
    return mean_train_mse / 10

print("Starting Phase 2: Optuna optimization for Adaptation Weights (10-fold Avg LOOCV)")
study_adapt = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_adapt.optimize(objective_adapt, n_trials=10000)

best_a = np.array([study_adapt.best_params[f'a_{col}'] for col in feature_cols])
print(f"Phase 2 Complete. Best 10-Fold Avg LOOCV MSE with Adaptation: {study_adapt.best_value:.4f}\n")

print("Optimized Similarity Weights (w):")
for name, weight in zip(feature_cols, best_w):
    print(f"  {name:15s}: {weight:.4f}")

print("\nOptimized Adaptation Weights (a):")
for name, weight in zip(feature_cols, best_a):
    print(f"  {name:15s}: {weight:.4f}")

metrics = {
    'train_uni': 0, 'test_uni': 0,
    'train_opt_w': 0, 'test_opt_w': 0,
    'train_adapt': 0, 'test_adapt': 0
}

uniform_w = np.ones(num_features) / num_features

print(f"{'Fold':<6} | {'Uniform Sim (Train / Test)':<28} | {'Opt Sim (Train / Test)':<26} | {'Opt Sim + Adapt (Train / Test)':<30}")

for i, fold in enumerate(folds):
    dist_tr_uni = np.sum(uniform_w * fold['diff_tr_sq'], axis=2)
    np.fill_diagonal(dist_tr_uni, np.inf)
    nn_tr_uni = np.argmin(dist_tr_uni, axis=1)
    train_uni_f = mean_squared_error(fold['y_tr'], fold['y_tr'][nn_tr_uni])
    
    dist_te_uni = np.sum(uniform_w * fold['diff_te_sq'], axis=2)
    nn_te_uni = np.argmin(dist_te_uni, axis=1)
    test_uni_f = mean_squared_error(fold['y_te'], fold['y_tr'][nn_te_uni])

    nn_tr_opt = fold['nn_idx_tr']
    train_opt_f = mean_squared_error(fold['y_tr'], fold['y_tr'][nn_tr_opt])
    
    dist_te_opt = np.sum(best_w * fold['diff_te_sq'], axis=2)
    nn_te_opt = np.argmin(dist_te_opt, axis=1)
    test_opt_f = mean_squared_error(fold['y_te'], fold['y_tr'][nn_te_opt])
    
    adapt_tr = np.sum(best_a * (fold['retrieved_X_tr'] - fold['X_tr']), axis=1)
    train_adapt_f = mean_squared_error(fold['y_tr'], fold['retrieved_y_tr'] + adapt_tr)
    
    retrieved_X_te = fold['X_tr'][nn_te_opt]
    retrieved_y_te = fold['y_tr'][nn_te_opt]
    adapt_te = np.sum(best_a * (retrieved_X_te - fold['X_te']), axis=1)
    test_adapt_f = mean_squared_error(fold['y_te'], retrieved_y_te + adapt_te)

    print(f"Fold {i:<2} | {train_uni_f:>10.4f} / {test_uni_f:<13.4f} | {train_opt_f:>9.4f} / {test_opt_f:<12.4f} | {train_adapt_f:>11.4f} / {test_adapt_f:<10.4f}")

    metrics['train_uni'] += train_uni_f
    metrics['test_uni'] += test_uni_f
    metrics['train_opt_w'] += train_opt_f
    metrics['test_opt_w'] += test_opt_f
    metrics['train_adapt'] += train_adapt_f
    metrics['test_adapt'] += test_adapt_f

for k in metrics: 
    metrics[k] /= 10

print("\n10-FOLD AVERAGE RESULTS")
print(f"Uniform Similarity:")
print(f"  Avg Train LOOCV MSE : {metrics['train_uni']:.4f}")
print(f"  Avg Test MSE        : {metrics['test_uni']:.4f}\n")

print(f"Optimized Similarity (Null Adaptation):")
print(f"  Avg Train LOOCV MSE : {metrics['train_opt_w']:.4f}")
print(f"  Avg Test MSE        : {metrics['test_opt_w']:.4f}\n")

print(f"Optimized Similarity + Optimized Adaptation:")
print(f"  Avg Train LOOCV MSE : {metrics['train_adapt']:.4f}")
print(f"  Avg Test MSE        : {metrics['test_adapt']:.4f}\n")

improvement = ((metrics['test_uni'] - metrics['test_adapt']) / metrics['test_uni']) * 100
print(f"Overall Unseen Test MSE Improvement (Uniform -> Full): {improvement:.2f}%")

Starting Phase 1: Optuna optimization for Similarity Weights (10-fold Avg LOOCV)
Phase 1 Complete. Best 10-Fold Avg LOOCV MSE: 9.3864

Starting Phase 2: Optuna optimization for Adaptation Weights (10-fold Avg LOOCV)
Phase 2 Complete. Best 10-Fold Avg LOOCV MSE with Adaptation: 8.8398

Optimized Similarity Weights (w):
  cylinders      : 0.0376
  displacement   : 0.4023
  horsepower     : 0.1876
  weight         : 0.2886
  acceleration   : 0.0027
  model_year     : 0.0813

Optimized Adaptation Weights (a):
  cylinders      : -1.5714
  displacement   : 0.4258
  horsepower     : 1.3029
  weight         : 1.4954
  acceleration   : 0.7524
  model_year     : -2.4221
Fold   | Uniform Sim (Train / Test)   | Opt Sim (Train / Test)     | Opt Sim + Adapt (Train / Test)
Fold 0  |    10.9288 / 9.3955        |   10.8261 / 7.1252       |     10.2234 / 6.1721    
Fold 1  |    11.9218 / 5.7335        |    9.6924 / 6.4327       |      8.9731 / 6.8523    
Fold 2  |    10.1555 / 11.1559       |    8.2127 

Simulating subsets as the size of footprint set of that fold for all the 10 folds in both 80-20 and 50-50 fashion

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import os
import networkx as nx
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

def get_train_footprint(X_tr, y_tr, sim_weights, adapt_weights):
    n_cases = len(X_tr)
    
    diff_tr = X_tr[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    dist = np.sum(sim_weights * (diff_tr ** 2), axis=2)
    np.fill_diagonal(dist, np.inf)
    nn_idx = np.argmin(dist, axis=1)
    
    best_X = X_tr[nn_idx]
    best_y = y_tr[nn_idx]
    
    adaptations = np.sum(adapt_weights * (best_X - X_tr), axis=1)
    preds = best_y + adaptations
    
    coverage = {i: set([i]) for i in range(n_cases)}
    reachability = {i: set([i]) for i in range(n_cases)}

    for i in range(n_cases):
        actual_mpg = y_tr[i]
        predicted_mpg = preds[i]
        best_nn_idx = nn_idx[i]
        
        if abs(predicted_mpg - actual_mpg) <= (0.05 * actual_mpg):
            coverage[best_nn_idx].add(i)
            reachability[i].add(best_nn_idx)

    rc = np.zeros(n_cases)
    for i in range(n_cases):
        for covered_idx in coverage[i]:
            if len(reachability[covered_idx]) > 0:
                rc[i] += 1.0 / len(reachability[covered_idx])

    related_sets = {i: coverage[i].union(reachability[i]) for i in range(n_cases)}
    G = nx.Graph()
    G.add_nodes_from(range(n_cases))
    
    for i in range(n_cases):
        for j in range(i + 1, n_cases):
            if len(related_sets[i].intersection(related_sets[j])) > 0:
                G.add_edge(i, j)

    global_footprint = set()
    for group in nx.connected_components(G):
        sorted_group = sorted(list(group), key=lambda x: (rc[x], len(coverage[x]), x), reverse=True)
        group_fp = set()
        changes = True
        while changes:
            changes = False
            for c_idx in sorted_group[:]:
                is_covered = any(c_idx in coverage[fp_idx] for fp_idx in group_fp)
                if not is_covered:
                    group_fp.add(c_idx)
                    sorted_group.remove(c_idx)
                    changes = True
        global_footprint.update(group_fp)

    return global_footprint

if __name__ == "__main__":
    
    best_w = np.array([0.038, 0.402, 0.188, 0.288, 0.003, 0.081], dtype=float)
    best_a = np.array([-1.571, 0.426, 1.303, 1.495, 0.752, -2.422], dtype=float)

    df = sns.load_dataset('mpg').dropna()
    feature_cols = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
    
    X_raw = df[feature_cols].values
    y_raw = df['mpg'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    
    save_dir = 'saved_subset_autompg_algorithm_1'
    os.makedirs(save_dir, exist_ok=True)
    np.random.seed(42)

    for fold_id, (train_idx, test_idx) in enumerate(kf.split(X_scaled)):
        print(f"\nPROCESSING FOLD {fold_id}")
        
        X_tr, y_tr = X_scaled[train_idx], y_raw[train_idx]
        n_train = len(X_tr)

        fp_indices = list(get_train_footprint(X_tr, y_tr, best_w, best_a))
        fp_size = len(fp_indices)
        print(f"Footprint Size: {fp_size} cases (out of {n_train} training cases)\n")
        
        fp_cases_concat = np.hstack((X_tr[fp_indices], y_tr[fp_indices].reshape(-1, 1)))
        np.save(f'{save_dir}/original_footprint_fold_{fold_id}.npy', fp_cases_concat)

        prob_scenario_1 = np.zeros(n_train)
        prob_scenario_1[fp_indices] = 0.8
        non_fp_indices = list(set(range(n_train)) - set(fp_indices))
        if non_fp_indices:
            prob_scenario_1[non_fp_indices] = 0.2
        prob_scenario_1 /= prob_scenario_1.sum()

        print("SCENARIO 1: Biased Selection (0.2 / 0.8)")
        for iteration in range(1, 11):
            subset_idx = np.random.choice(n_train, size=fp_size, replace=False, p=prob_scenario_1)
            subset_cases_concat = np.hstack((X_tr[subset_idx], y_tr[subset_idx].reshape(-1, 1)))
            np.save(f'{save_dir}/original_footprint_fold_{fold_id}_scenario_1_iter_{iteration}.npy', subset_cases_concat)
            
            missing_cases = len(set(fp_indices) - set(subset_idx))
            print(f"  Iteration {iteration:2d} | Missing Original FP Cases: {missing_cases}")

        prob_scenario_2 = np.ones(n_train) * 0.5
        prob_scenario_2 /= prob_scenario_2.sum()

        print("\nSCENARIO 2: Uniform Random Selection (0.5)")
        for iteration in range(1, 11):
            subset_idx = np.random.choice(n_train, size=fp_size, replace=False, p=prob_scenario_2)
            subset_cases_concat = np.hstack((X_tr[subset_idx], y_tr[subset_idx].reshape(-1, 1)))
            np.save(f'{save_dir}/original_footprint_fold_{fold_id}_scenario_2_iter_{iteration}.npy', subset_cases_concat)
            
            missing_cases = len(set(fp_indices) - set(subset_idx))
            print(f"  Iteration {iteration:2d} | Missing Original FP Cases: {missing_cases}")


PROCESSING FOLD 0
Footprint Size: 265 cases (out of 352 training cases)

SCENARIO 1: Biased Selection (0.2 / 0.8)
  Iteration  1 | Missing Original FP Cases: 36
  Iteration  2 | Missing Original FP Cases: 31
  Iteration  3 | Missing Original FP Cases: 34
  Iteration  4 | Missing Original FP Cases: 32
  Iteration  5 | Missing Original FP Cases: 33
  Iteration  6 | Missing Original FP Cases: 30
  Iteration  7 | Missing Original FP Cases: 36
  Iteration  8 | Missing Original FP Cases: 35
  Iteration  9 | Missing Original FP Cases: 36
  Iteration 10 | Missing Original FP Cases: 35

SCENARIO 2: Uniform Random Selection (0.5)
  Iteration  1 | Missing Original FP Cases: 65
  Iteration  2 | Missing Original FP Cases: 69
  Iteration  3 | Missing Original FP Cases: 67
  Iteration  4 | Missing Original FP Cases: 64
  Iteration  5 | Missing Original FP Cases: 64
  Iteration  6 | Missing Original FP Cases: 66
  Iteration  7 | Missing Original FP Cases: 63
  Iteration  8 | Missing Original FP Cases

Evaluation and statistics after adding the missing cases of the footprint sets (Between original subset and added subset)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import os
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import ttest_rel

def evaluate_subset(X_subset, y_subset, X_te, y_te, sim_weights, adapt_weights):
    diff = X_te[:, np.newaxis, :] - X_subset[np.newaxis, :, :]
    dist = np.sum(sim_weights * (diff ** 2), axis=2)
    nn_idx = np.argmin(dist, axis=1)
    
    best_X = X_subset[nn_idx]
    best_y = y_subset[nn_idx]
    
    adaptations = np.sum(adapt_weights * (best_X - X_te), axis=1)
    preds = best_y + adaptations
    
    return mean_squared_error(y_te, preds)

def get_missing_cases(fp_cases, subset_cases):
    missing = []
    for fp_row in fp_cases:
        if not np.any(np.all(np.isclose(subset_cases, fp_row, atol=1e-5), axis=1)):
            missing.append(fp_row)
    return np.array(missing) if len(missing) > 0 else np.empty((0, fp_cases.shape[1]))

if __name__ == "__main__":
    
    best_w = np.array([0.038, 0.402, 0.188, 0.288, 0.003, 0.081], dtype=float)
    best_a = np.array([-1.571, 0.426, 1.303, 1.495, 0.752, -2.422], dtype=float)

    df = sns.load_dataset('mpg').dropna()
    feature_cols = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
    
    X_raw = df[feature_cols].values
    y_raw = df['mpg'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    save_dir = 'saved_subset_autompg_algorithm_1'
    
    global_better = 0
    global_same = 0
    global_worse = 0
    global_total = 0
    
    global_unadded_mses = []
    global_added_mses = []
    global_added_counts = []
    global_added_pcts = []
    
    global_scen1_unadded = []
    global_scen1_added = []
    global_scen1_added_counts = []
    global_scen1_added_pcts = []
    
    global_scen2_unadded = []
    global_scen2_added = []
    global_scen2_added_counts = []
    global_scen2_added_pcts = []

    global_scen1_better = 0
    global_scen1_same = 0
    global_scen1_worse = 0

    global_scen2_better = 0
    global_scen2_same = 0
    global_scen2_worse = 0

    for fold_id, (train_idx, test_idx) in enumerate(kf.split(X_scaled)):
        print(f"\nEVALUATING FOLD {fold_id}")
        
        X_tr_complete, y_tr_complete = X_scaled[train_idx], y_raw[train_idx]
        X_te, y_te = X_scaled[test_idx], y_raw[test_idx]

        complete_mse = evaluate_subset(X_tr_complete, y_tr_complete, X_te, y_te, best_w, best_a)
        print(f"Complete Train Data Test MSE: {complete_mse:.4f}")

        fp_path = f'{save_dir}/original_footprint_fold_{fold_id}.npy'
        if not os.path.exists(fp_path):
            print(f"Missing {fp_path}, skipping fold...")
            continue
            
        fp_data = np.load(fp_path)
        print(f"Original Footprint Size: {len(fp_data)}")
        
        X_fp, y_fp = fp_data[:, :-1], fp_data[:, -1]
        fp_mse = evaluate_subset(X_fp, y_fp, X_te, y_te, best_w, best_a)
        print(f"Original Footprint Test MSE : {fp_mse:.4f}")
        
        fold_better = 0
        fold_same = 0
        fold_worse = 0
        
        fold_unadded_mses = []
        fold_added_mses = []
        fold_added_counts = []
        fold_added_pcts = []

        for scenario in [1, 2]:
            print(f"\nSCENARIO {scenario}")
            scenario_unadded_mses = []
            scenario_added_mses = []
            scenario_added_counts = []
            scenario_added_pcts = []
            
            for iteration in range(1, 11):
                subset_path = f'{save_dir}/original_footprint_fold_{fold_id}_scenario_{scenario}_iter_{iteration}.npy'
                
                if not os.path.exists(subset_path):
                    continue
                
                subset_data = np.load(subset_path)
                X_sub, y_sub = subset_data[:, :-1], subset_data[:, -1]
                
                unadded_mse = evaluate_subset(X_sub, y_sub, X_te, y_te, best_w, best_a)
    
                missing_cases = get_missing_cases(fp_data, subset_data)
                added_count = len(missing_cases)
                
                added_pct = (added_count / len(fp_data)) * 100.0 if len(fp_data) > 0 else 0.0
                
                if added_count > 0:
                    added_data = np.vstack((subset_data, missing_cases))
                    X_add, y_add = added_data[:, :-1], added_data[:, -1]
                    added_mse = evaluate_subset(X_add, y_add, X_te, y_te, best_w, best_a)
                else:
                    added_data = subset_data
                    added_mse = unadded_mse

                updated_path = f'{save_dir}/original_footprint_fold_{fold_id}_scenario_{scenario}_iter_{iteration}_updated.npy'
                np.save(updated_path, added_data)

                scenario_unadded_mses.append(unadded_mse)
                scenario_added_mses.append(added_mse)
                scenario_added_counts.append(added_count)
                scenario_added_pcts.append(added_pct)

                if np.isclose(added_mse, unadded_mse, atol=1e-8):
                    fold_same += 1
                    if scenario == 1: global_scen1_same += 1
                    elif scenario == 2: global_scen2_same += 1
                elif added_mse < unadded_mse:
                    fold_better += 1
                    if scenario == 1: global_scen1_better += 1
                    elif scenario == 2: global_scen2_better += 1
                else:
                    fold_worse += 1
                    if scenario == 1: global_scen1_worse += 1
                    elif scenario == 2: global_scen2_worse += 1
                
                print(f"Iter {iteration:2d} | Unadded MSE: {unadded_mse:6.4f} | "
                      f"Missing FP Added: {added_count:2d} ({added_pct:5.2f}%) -> New Size: {len(subset_data)+added_count:3d} | "
                      f"Added MSE: {added_mse:6.4f}")
            
            if len(scenario_unadded_mses) > 0:
                scen_avg_unadded = np.mean(scenario_unadded_mses)
                scen_avg_added = np.mean(scenario_added_mses)
                scen_avg_cases = np.mean(scenario_added_counts)
                scen_avg_pcts = np.mean(scenario_added_pcts)
                print(f"Scenario {scenario} Avg Unadded MSE: {scen_avg_unadded:.4f} | Avg Added MSE: {scen_avg_added:.4f}")
                print(f"Scenario {scenario} Avg Cases Added: {scen_avg_cases:.2f} ({scen_avg_pcts:.2f}%)")
                
                if len(scenario_unadded_mses) > 1:
                    t_stat, p_val = ttest_rel(scenario_unadded_mses, scenario_added_mses)
                    print(f"Scenario {scenario} Paired T-Test: t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")

                fold_unadded_mses.extend(scenario_unadded_mses)
                fold_added_mses.extend(scenario_added_mses)
                fold_added_counts.extend(scenario_added_counts)
                fold_added_pcts.extend(scenario_added_pcts)
                
                if scenario == 1:
                    global_scen1_unadded.extend(scenario_unadded_mses)
                    global_scen1_added.extend(scenario_added_mses)
                    global_scen1_added_counts.extend(scenario_added_counts)
                    global_scen1_added_pcts.extend(scenario_added_pcts)
                elif scenario == 2:
                    global_scen2_unadded.extend(scenario_unadded_mses)
                    global_scen2_added.extend(scenario_added_mses)
                    global_scen2_added_counts.extend(scenario_added_counts)
                    global_scen2_added_pcts.extend(scenario_added_pcts)

        print(f"\nFOLD {fold_id} SUMMARY:")
        if len(fold_unadded_mses) > 0:
            fold_avg_unadded = np.mean(fold_unadded_mses)
            fold_avg_added = np.mean(fold_added_mses)
            fold_avg_cases = np.mean(fold_added_counts)
            fold_avg_pcts = np.mean(fold_added_pcts)
            print(f"Fold Avg Unadded MSE: {fold_avg_unadded:.4f} | Fold Avg Added MSE: {fold_avg_added:.4f}")
            print(f"Fold Avg Cases Added: {fold_avg_cases:.2f} ({fold_avg_pcts:.2f}%)")
            
            if len(fold_unadded_mses) > 1:
                t_stat, p_val = ttest_rel(fold_unadded_mses, fold_added_mses)
                print(f"Fold Paired T-Test: t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")

            global_unadded_mses.extend(fold_unadded_mses)
            global_added_mses.extend(fold_added_mses)
            global_added_counts.extend(fold_added_counts)
            global_added_pcts.extend(fold_added_pcts)
            
        print(f"Better: {fold_better} | Same: {fold_same} | Worse: {fold_worse} (Out of {fold_better + fold_same + fold_worse})")
        
        global_better += fold_better
        global_same += fold_same
        global_worse += fold_worse
        global_total += (fold_better + fold_same + fold_worse)

    print(f"\nGLOBAL SUMMARY")

    print(f"\n[GLOBAL SCENARIO 1]")
    if len(global_scen1_unadded) > 0:
        print(f"Avg Unadded MSE         : {np.mean(global_scen1_unadded):.4f}")
        print(f"Avg Added MSE           : {np.mean(global_scen1_added):.4f}")
        print(f"Avg Cases Added         : {np.mean(global_scen1_added_counts):.2f} ({np.mean(global_scen1_added_pcts):.2f}%)")
        print(f"MSE Decreased (Improved): {global_scen1_better}")
        print(f"MSE Remained the Same   : {global_scen1_same}")
        print(f"MSE Increased (Worsened): {global_scen1_worse}")
        if len(global_scen1_unadded) > 1:
            t_stat, p_val = ttest_rel(global_scen1_unadded, global_scen1_added)
            print(f"Paired T-Test           : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
    else:
        print("No data for Scenario 1.")

    print(f"\n[GLOBAL SCENARIO 2]")
    if len(global_scen2_unadded) > 0:
        print(f"Avg Unadded MSE         : {np.mean(global_scen2_unadded):.4f}")
        print(f"Avg Added MSE           : {np.mean(global_scen2_added):.4f}")
        print(f"Avg Cases Added         : {np.mean(global_scen2_added_counts):.2f} ({np.mean(global_scen2_added_pcts):.2f}%)")
        print(f"MSE Decreased (Improved): {global_scen2_better}")
        print(f"MSE Remained the Same   : {global_scen2_same}")
        print(f"MSE Increased (Worsened): {global_scen2_worse}")
        if len(global_scen2_unadded) > 1:
            t_stat, p_val = ttest_rel(global_scen2_unadded, global_scen2_added)
            print(f"Paired T-Test           : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
    else:
        print("No data for Scenario 2.")

    print(f"\n[ENTIRE PROCESS (Overall)]")
    if len(global_unadded_mses) > 0:
        global_avg_unadded = np.mean(global_unadded_mses)
        global_avg_added = np.mean(global_added_mses)
        print(f"Avg Unadded MSE         : {global_avg_unadded:.4f}")
        print(f"Avg Added MSE           : {global_avg_added:.4f}")
        print(f"Avg Cases Added         : {np.mean(global_added_counts):.2f} ({np.mean(global_added_pcts):.2f}%)")
        
        if len(global_unadded_mses) > 1:
            t_stat, p_val = ttest_rel(global_unadded_mses, global_added_mses)
            print(f"Paired T-Test           : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
            
    print(f"MSE Decreased (Improved): {global_better}")
    print(f"MSE Remained the Same   : {global_same}")
    print(f"MSE Increased (Worsened): {global_worse}")
    print(f"Total Evaluated Subsets : {global_total}")


EVALUATING FOLD 0
Complete Train Data Test MSE: 6.1723
Original Footprint Size: 265
Original Footprint Test MSE : 7.4168

SCENARIO 1
Iter  1 | Unadded MSE: 6.9599 | Missing FP Added: 36 (13.58%) -> New Size: 301 | Added MSE: 7.2124
Iter  2 | Unadded MSE: 7.3159 | Missing FP Added: 31 (11.70%) -> New Size: 296 | Added MSE: 7.9608
Iter  3 | Unadded MSE: 7.8390 | Missing FP Added: 34 (12.83%) -> New Size: 299 | Added MSE: 7.3978
Iter  4 | Unadded MSE: 8.8060 | Missing FP Added: 32 (12.08%) -> New Size: 297 | Added MSE: 8.1532
Iter  5 | Unadded MSE: 7.2218 | Missing FP Added: 33 (12.45%) -> New Size: 298 | Added MSE: 7.1802
Iter  6 | Unadded MSE: 7.5244 | Missing FP Added: 30 (11.32%) -> New Size: 295 | Added MSE: 7.8859
Iter  7 | Unadded MSE: 5.5858 | Missing FP Added: 36 (13.58%) -> New Size: 301 | Added MSE: 6.6503
Iter  8 | Unadded MSE: 8.1637 | Missing FP Added: 35 (13.21%) -> New Size: 300 | Added MSE: 7.3556
Iter  9 | Unadded MSE: 7.6759 | Missing FP Added: 36 (13.58%) -> New Size: